# 02. Vertex AI 기반 LLM 리뷰 분석

`01_preprocess_reviews.ipynb`전처리 파일에서 만든 후보 데이터를 받아서\
분석 조건 필터링과 샘플링을 적용한 뒤 **PydanticAI + Vertex AI 기반 Gemini**로 Steam 리뷰 감성·이슈 분석을 실행한다.


## 역할
1. 전처리 완료 후보 리뷰를 불러온다.
2. 장르, 출시일, 언어, 출시 구간, 리뷰 신뢰도 조건을 적용한다.
3. 게임별 리뷰 수를 제한하고 Steam 긍정/부정 라벨을 균형 있게 샘플링한다.
4. LLM 입력 파일을 저장한다.
5. Vertex AI 기반 Gemini 모델로 리뷰별 감정과 이슈를 분석한다.
6. 보고서에서 사용할 CSV 산출물을 저장한다.

## 이 노트북에서 사용하는 주요 입력
- `llm_preprocessed_reviews.csv`
- `llm_candidate_game_summary.csv`

## 이 노트북에서 생성하는 주요 산출물
- `llm_input_reviews.csv`
- `llm_review_analysis_result.csv`
- `llm_issue_tags_flat.csv`

## 분석 흐름

```
01_preprocess_reviews.ipynb 산출물
llm_preprocessed_reviews.csv
        ↓
분석 조건 필터링
장르 / 출시일 / 언어 / 출시 구간 / 리뷰 신뢰도
        ↓
게임별 리뷰 샘플링
recent / random / balanced_by_steam_label
        ↓
LLM 입력 파일 저장
llm_input_reviews.csv
        ↓
Vertex AI Gemini 호출
PydanticAI 구조화 출력
        ↓
리뷰 단위 결과 저장
llm_review_analysis_result.csv
        ↓
이슈 태그 펼치기
llm_issue_tags_flat.csv
```


# 0. 환경설정

In [1]:
# ============================================================
# 기본 라이브러리
# ============================================================
import os
import ast
import json
import time
import asyncio
import platform
from pathlib import Path
from typing import List, Literal, Optional
from datetime import datetime

# ============================================================
# 데이터 분석용 라이브러리
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# ============================================================
# 환경변수 / 진행률 / LLM 출력 스키마 관련 라이브러리
# ============================================================
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tqdm.auto import tqdm
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel, GoogleModelSettings
from pydantic_ai.providers.google import GoogleProvider
from google import genai
from google.genai import types



# ============================================================
# 한글 폰트 설정
# ============================================================
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":  # macOS
    plt.rcParams["font.family"] = "AppleGothic"
else:  # Linux
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (12, 6)

# pandas 출력 옵션
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

c:\Users\joon5\Documents\github\steam-indie-game-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 경로 설정

In [2]:
# 프로젝트 루트 직접 지정
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정합니다.
ROOT = Path(r"C:\Users\joon5\Documents\github\steam-indie-game-analysis")

# ============================================================
# 전처리 산출물 폴더
# ============================================================
PREPROCESS_RUN_NAME = "preprocess_llm"
PREPROCESS_OUTPUT_DIR = ROOT / "data" / "outputs" / PREPROCESS_RUN_NAME

# 전처리 파일에서 만든 후보 데이터
PREPROCESSED_REVIEWS_PATH = PREPROCESS_OUTPUT_DIR / "llm_preprocessed_reviews.csv"
CANDIDATE_GAME_SUMMARY_PATH = PREPROCESS_OUTPUT_DIR / "llm_candidate_game_summary.csv"

# ============================================================
# LLM 분석 결과 저장 폴더
# ============================================================
RUN_NAME = "postlaunch_zoonomaly_v1"
OUTPUT_DIR = ROOT / "data" / "outputs" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# LLM 실행 직전 산출물
LLM_INPUT_PATH = OUTPUT_DIR / "llm_input_reviews.csv"
FILTER_LOG_PATH = OUTPUT_DIR / "llm_input_filter_log.csv"
SAMPLE_SUMMARY_PATH = OUTPUT_DIR / "llm_input_sample_summary.csv"

# LLM 분석 산출물
CHECKPOINT_PATH = OUTPUT_DIR / "llm_review_analysis_checkpoint.json"
RESULT_JSON_PATH = OUTPUT_DIR / "llm_review_analysis_result.json"
RESULT_CSV_PATH = OUTPUT_DIR / "llm_review_analysis_result.csv"
ISSUE_TAG_FLAT_PATH = OUTPUT_DIR / "llm_issue_tags_flat.csv"

# 선택 산출물
GAME_SUMMARY_PATH = OUTPUT_DIR / "llm_game_summary.csv"
ISSUE_PRIORITY_PATH = OUTPUT_DIR / "llm_issue_priority_summary.csv"

print("ROOT 존재:", ROOT.exists())
print("전처리 폴더 존재:", PREPROCESS_OUTPUT_DIR.exists())
print("전처리 리뷰 파일 존재:", PREPROCESSED_REVIEWS_PATH.exists())
print("전처리 게임 요약 파일 존재:", CANDIDATE_GAME_SUMMARY_PATH.exists())
print("결과 저장 폴더 존재:", OUTPUT_DIR.exists())

print("\n프로젝트 루트:", ROOT)
print("전처리 후보 데이터:", PREPROCESSED_REVIEWS_PATH)
print("전처리 게임 요약:", CANDIDATE_GAME_SUMMARY_PATH)
print("결과 저장 폴더:", OUTPUT_DIR)


ROOT 존재: True
전처리 폴더 존재: True
전처리 리뷰 파일 존재: True
전처리 게임 요약 파일 존재: True
결과 저장 폴더 존재: True

프로젝트 루트: C:\Users\joon5\Documents\github\steam-indie-game-analysis
전처리 후보 데이터: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\preprocess_llm\llm_preprocessed_reviews.csv
전처리 게임 요약: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\preprocess_llm\llm_candidate_game_summary.csv
결과 저장 폴더: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_zoonomaly_v1


## Vertex AI 설정

In [3]:
# ============================================================
# Vertex AI / PydanticAI 설정
# ============================================================
# 이 셀은 실제 LLM 호출을 위한 Google Cloud 프로젝트, location, 모델명을 설정한다.

# .env 파일에 저장된 Vertex AI 설정을 현재 Python 환경으로 불러온다.
load_dotenv()
load_dotenv(ROOT / ".env")

# Google Cloud 프로젝트 ID를 읽는다.
GOOGLE_CLOUD_PROJECT = (
    os.getenv("GOOGLE_CLOUD_PROJECT")
    or os.getenv("VERTEX_PROJECT_ID")
    or os.getenv("GCP_PROJECT_ID")
)

# Vertex AI location을 읽는다.
GOOGLE_CLOUD_LOCATION = (
    os.getenv("GOOGLE_CLOUD_LOCATION")
    or os.getenv("VERTEX_LOCATION")
    or "global"
)

# 사용할 Gemini 모델명
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite-preview")

# 프로젝트 ID가 있으면 Vertex AI Provider와 Gemini 모델 객체를 생성한다.
if GOOGLE_CLOUD_PROJECT:
    vertex_provider = GoogleProvider(
        vertexai=True,
        project=GOOGLE_CLOUD_PROJECT,
        location=GOOGLE_CLOUD_LOCATION,
    )

    vertex_model = GoogleModel(
        GEMINI_MODEL,
        provider=vertex_provider,
    )

    print("Vertex AI project:", GOOGLE_CLOUD_PROJECT)
    print("Vertex AI location:", GOOGLE_CLOUD_LOCATION)
    print("Gemini model:", GEMINI_MODEL)
    print("Vertex 모델 생성: O")
else:
    # RUN_LLM=False로 기존 결과만 읽어 후처리할 때는 프로젝트 ID가 없어도 노트북을 계속 볼 수 있게 한다.
    # 단, 실제 LLM 실행 전에는 반드시 .env 또는 환경변수에 GOOGLE_CLOUD_PROJECT를 설정해야 한다.
    vertex_provider = None
    vertex_model = None

    print("Vertex AI project: 미설정")
    print("Vertex 모델 생성: X")
    print("실제 LLM 실행 전 .env에 GOOGLE_CLOUD_PROJECT를 설정하세요.")


Vertex AI project: gen-lang-client-0587784564
Vertex AI location: global
Gemini model: gemini-3.1-flash-lite-preview
Vertex 모델 생성: O


# 1. 분석/샘플링/LLM 실행 설정

|   appid | 게임명                            | 스팀 링크 |
| ------: | --------------------------------- | ------------------------------------------------------------------------------- |
| 1466060 | Tainted Grail: The Fall of Avalon | https://store.steampowered.com/app/1466060/Tainted_Grail_The_Fall_of_Avalon/    |
| 2101890 | Zoonomaly                         | https://store.steampowered.com/app/2101890/Zoonomaly/                           |
| 2488510 | FatalZone                         | https://store.steampowered.com/app/2488510/FatalZone/                           |

Tainted Grail: The Fall of Avalon 
Zoonomaly  
Drova - Forsaken Kin

|   appid| 게임                                   |        출시일 |  전체 리뷰 | 전체 부정 리뷰 | 현재 파일 내 분석 가능 리뷰 | 현재 파일 내 부정 리뷰 | 스팀 링크 |
| :----- | ------------------------------------- | -------- | ----- | ------- | --------------- | ------------ | ------ | 
| 1466060| **Tainted Grail: The Fall of Avalon** | 2025-05-23 | 11,480 |    1,437 |           11,982 |         1,632 | https://store.steampowered.com/app/1466060/Tainted_Grail_The_Fall_of_Avalon/    |
| 1585180 | **Drova - Forsaken Kin**             | 2024-10-15 |  6,908 |      324 |            3,960 |           280 | https://store.steampowered.com/app/1585180/Drova__Forsaken_Kin/ |
| 2101890 | **Zoonomaly**                        | 2024-03-06 |    778 |      183 |              296 |            84 | https://store.steampowered.com/app/2101890/Zoonomaly/                           |



In [4]:
# ============================================================
# 실행 여부
# ============================================================
RUN_LLM = True
RESET_CHECKPOINT = True
RUN_CHECK_CELLS = True

# ============================================================
# 분석 대상 설정
# ============================================================
TARGET_GENRE = None

ANALYSIS_FILTERS = {
    "appids": [2101890],
    "game_name_contains": [],

    "genres": [] if TARGET_GENRE is None else [TARGET_GENRE],
    "categories": [],
    "tags": [],
    "release_date_from": "2024-01-01",
    "release_date_to": None,

    "languages": ["english"],
    "steam_labels": ["positive", "negative"],
    "review_date_from": None,
    "review_date_to": None,

    # 전체 기간 리뷰 사용
    "release_periods": [],

    "steam_purchase_only": True,
    "exclude_received_for_free": True,
    "exclude_early_access_reviews": True,

    "meaningful_review_only": True,
}

# ============================================================
# 샘플링 설정
# ============================================================
TEST_N = None
RANDOM_STATE = 42

# Zoonomaly는 리뷰 수가 적으므로 가능한 의미 있는 리뷰를 최대한 사용
# 현재 분석 가능 리뷰가 약 296개라서 300으로 설정하면 사실상 전체 분석에 가까움
REVIEWS_PER_GAME = 300
MAX_TOTAL_REVIEWS = 300

# 부정 리뷰가 70%까지 충분하지 않으므로,
# 가능한 부정 리뷰를 최대한 포함하고 나머지는 긍정 리뷰로 보완
NEGATIVE_SAMPLE_RATIO = 0.7
SAMPLE_MODE = "negative_heavy_by_steam_label"

# ============================================================
# LLM 입력 텍스트 설정
# ============================================================
MIN_REVIEW_LEN = 40
MAX_REVIEW_CHARS = 1200

# ============================================================
# LLM 호출 설정
# ============================================================
BATCH_SIZE = 3
MAX_CONCURRENT = 1
MAX_RETRIES = 3
REQUEST_SLEEP_SEC = 1
CHUNK_SIZE = MAX_CONCURRENT * 3

# ============================================================
# 저장 옵션
# ============================================================
SAVE_RESULT_JSON = True
SAVE_OPTIONAL_SUMMARY_FILES = False

# ============================================================
# 비용 추정 옵션
# 실제 Vertex AI 과금과 다를 수 있으므로 참고용으로만 사용한다.
# ============================================================
# 아래 단가는 대략적인 참고용이다.
# 실제 비용 확인은 Google Cloud Billing / Vertex AI pricing 기준으로 확인한다.
INPUT_PRICE_PER_1M = 0.30
OUTPUT_PRICE_PER_1M = 2.50
USD_TO_KRW = 1500

print("RUN_LLM:", RUN_LLM)
print("TARGET_GENRE:", TARGET_GENRE)
print("SAMPLE_MODE:", SAMPLE_MODE)
print("REVIEWS_PER_GAME:", REVIEWS_PER_GAME)
print("MAX_TOTAL_REVIEWS:", MAX_TOTAL_REVIEWS)


RUN_LLM: True
TARGET_GENRE: None
SAMPLE_MODE: negative_heavy_by_steam_label
REVIEWS_PER_GAME: 300
MAX_TOTAL_REVIEWS: 300


# 2. 공통 함수

In [5]:
# pandas/numpy/Pydantic 객체를 JSON/CSV 저장 가능한 기본 타입으로 변환한다.
def to_serializable(obj):
    """JSON 저장이 어려운 pandas/numpy/Pydantic 타입을 기본 Python 타입으로 변환한다."""
    if isinstance(obj, BaseModel):
        return to_serializable(obj.model_dump())
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, datetime):
        return obj.isoformat()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj

# checkpoint 저장/불러오기 함수
def load_checkpoint(path=CHECKPOINT_PATH):
    """이전 LLM 분석 checkpoint를 불러온다."""
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_checkpoint(results, path=CHECKPOINT_PATH):
    """현재까지의 LLM 분석 결과를 checkpoint JSON으로 저장한다."""
    safe_results = to_serializable(results)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_results, f, ensure_ascii=False, indent=2)


def load_existing_results():
    """RUN_LLM=False일 때 기존 결과를 읽는다."""
    if RESULT_JSON_PATH.exists():
        with open(RESULT_JSON_PATH, "r", encoding="utf-8") as f:
            return json.load(f)

    return load_checkpoint(CHECKPOINT_PATH)


# 비용/토큰 사용량 확인 함수
def print_cost_report(input_tokens, output_tokens, requests, checkpoint_count, to_process_count):
    """토큰 사용량과 예상 비용을 출력한다."""
    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_1M
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
    total_cost = input_cost + output_cost

    print("=" * 60)
    print("토큰 사용량 / 예상 비용")
    print("=" * 60)
    print(f"요청 수: {requests:,}")
    print(f"checkpoint에서 불러온 리뷰 수: {checkpoint_count:,}")
    print(f"이번 실행에서 새로 처리할 리뷰 수: {to_process_count:,}")
    print(f"입력 토큰: {input_tokens:,}")
    print(f"출력 토큰: {output_tokens:,}")
    print(f"예상 비용(USD): ${total_cost:,.6f}")
    print(f"예상 비용(KRW): ₩{total_cost * USD_TO_KRW:,.0f}")
    print("=" * 60)

# PydanticAI 사용량 추출 함수
def extract_usage_tokens(result):
    """
    PydanticAI 결과 객체에서 토큰 사용량을 안전하게 추출한다.

    PydanticAI/모델 버전에 따라 usage 속성명이 조금 다를 수 있어
    여러 후보 이름을 순서대로 확인한다.
    """
    input_tokens = 0
    output_tokens = 0

    try:
        usage = result.usage()

        input_tokens = (
            getattr(usage, "input_tokens", None)
            or getattr(usage, "request_tokens", None)
            or getattr(usage, "prompt_tokens", None)
            or 0
        )

        output_tokens = (
            getattr(usage, "output_tokens", None)
            or getattr(usage, "response_tokens", None)
            or getattr(usage, "completion_tokens", None)
            or 0
        )

    except Exception:
        pass

    return int(input_tokens or 0), int(output_tokens or 0)

# 필터링/문자열 검색 헬퍼 함수
def print_filter_step(log_rows, step, before, after):
    """필터링 단계별 행 수 변화를 기록한다."""
    removed = before - after
    removed_rate = removed / before if before else 0
    log_rows.append({
        "step": step,
        "before_rows": before,
        "after_rows": after,
        "removed_rows": removed,
        "removed_rate": removed_rate,
    })
    print(f"{step}: {before:,} -> {after:,} / 제거 {removed:,} ({removed_rate:.2%})")


def contains_any_text(text, keywords):
    """문자열에 키워드 중 하나라도 포함되어 있는지 확인한다."""
    if not keywords:
        return True
    if pd.isna(text):
        return False
    text = str(text).lower()
    return any(str(keyword).lower() in text for keyword in keywords)


def pydantic_list_to_dicts(items):
    """Pydantic 객체 리스트를 CSV/JSON 저장 가능한 dict 리스트로 변환한다."""
    if items is None:
        return []
    if not isinstance(items, list):
        return []

    converted = []
    for item in items:
        if isinstance(item, BaseModel):
            converted.append(item.model_dump())
        elif isinstance(item, dict):
            converted.append(item)
    return converted


# 3. 전처리 후보 데이터 로드

In [6]:
# 전처리 후보 데이터 로드
df_candidates = pd.read_csv(PREPROCESSED_REVIEWS_PATH)

for col in ["review_datetime", "release_date"]:
    if col in df_candidates.columns:
        df_candidates[col] = pd.to_datetime(df_candidates[col], errors="coerce")

if "recommendationid" in df_candidates.columns:
    df_candidates["recommendationid"] = df_candidates["recommendationid"].astype(str)

print("전처리 후보 리뷰 수:", len(df_candidates))
print("전처리 후보 게임 수:", df_candidates["appid"].nunique())
display(df_candidates.head())


전처리 후보 리뷰 수: 166875
전처리 후보 게임 수: 188


,recommendationid,appid,game_name,language,review_datetime,release_date,days_from_release,release_period,release_period_detail,steam_label_text,voted_up,playtime_at_review_hours,playtime_forever_hours,playtime_stage,votes_up,weighted_vote_score,steam_purchase,received_for_free,written_during_early_access,genres_text,categories_text,top_steam_tags_text,price,price_group,review_text_clean,review_len,is_meaningful_review,meaningless_reason
0,18698790,324470,SinaRun,french,2015-10-26 18:10:33,2025-11-03,-3661.0,pre_release,pre_release,positive,True,1.250000,3.483333,early,2,0.523810,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,good game for this price,24,False,too_short
1,18699465,324470,SinaRun,english,2015-10-26 18:53:36,2025-11-03,-3661.0,pre_release,pre_release,positive,True,0.216667,0.216667,very_early,1,0.421372,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effect is too excessive \nSo level of difficulty is too hard for beginner ...",119,True,meaningful
2,18699648,324470,SinaRun,english,2015-10-26 19:03:37,2025-11-03,-3661.0,pre_release,pre_release,positive,True,12.666667,13.616667,late,5,0.500076,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,"this game is like a zen-garden, I love it! \n\npros:\n-it's very relaxing\n-good controls\n-awesome leveldesign\n-re...",387,True,meaningful
3,18700348,324470,SinaRun,english,2015-10-26 19:52:39,2025-11-03,-3661.0,pre_release,pre_release,positive,True,0.916667,7.483333,early,16,0.637511,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,Ever played Bhop? Surf? If so this games mechanics will feel Instantly similar too you. This game gives you such a r...,1100,True,meaningful
4,18701774,324470,SinaRun,english,2015-10-26 21:32:24,2025-11-03,-3661.0,pre_release,pre_release,positive,True,6.416667,8.666667,mid,4,0.495810,True,False,True,"Indie, Racing","Single-player, Steam Achievements, Steam Trading Cards, Camera Comfort, Custom Volume Controls, Mouse Only Option, P...","Indie, Racing, Early Access, Parkour, First-Person",3.99,0-5,It's Lit,8,False,too_short


# 4. 분석 조건 필터링

In [7]:
# ANALYSIS_FILTERS에 입력한 조건을 실제 후보 데이터에 적용한다.
def apply_analysis_filters(df, filters):
    """LLM 실행 파일에서 분석 목적에 맞는 필터를 적용한다."""
    filtered = df.copy()
    log_rows = []

    # 0. 최소 리뷰 길이 필터
    if "review_len" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["review_len"] >= MIN_REVIEW_LEN]
        print_filter_step(log_rows, "최소 리뷰 길이 필터", before, len(filtered))

    # 0-1. 의미 있는 리뷰 필터
    if filters.get("meaningful_review_only") is True and "is_meaningful_review" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["is_meaningful_review"] == True]
        print_filter_step(log_rows, "의미 있는 리뷰 필터", before, len(filtered))

    # 1. 게임 직접 지정 필터
    if filters.get("appids"):
        before = len(filtered)
        appids = [int(x) for x in filters["appids"]]
        filtered = filtered[filtered["appid"].isin(appids)]
        print_filter_step(log_rows, "appid 필터", before, len(filtered))

    if filters.get("game_name_contains"):
        before = len(filtered)
        keywords = [str(x).lower() for x in filters["game_name_contains"]]
        filtered = filtered[filtered["game_name"].fillna("").str.lower().apply(lambda x: any(k in x for k in keywords))]
        print_filter_step(log_rows, "게임명 키워드 필터", before, len(filtered))

    # 2. 게임 속성 필터
    # 장르/카테고리/태그 텍스트에 지정한 키워드가 포함되는지 확인한다.
    if filters.get("genres"):
        before = len(filtered)
        filtered = filtered[filtered["genres_text"].apply(lambda x: contains_any_text(x, filters["genres"]))]
        print_filter_step(log_rows, "장르 필터", before, len(filtered))

    if filters.get("categories"):
        before = len(filtered)
        filtered = filtered[filtered["categories_text"].apply(lambda x: contains_any_text(x, filters["categories"]))]
        print_filter_step(log_rows, "카테고리 필터", before, len(filtered))

    if filters.get("tags"):
        before = len(filtered)
        filtered = filtered[filtered["top_steam_tags_text"].apply(lambda x: contains_any_text(x, filters["tags"]))]
        print_filter_step(log_rows, "태그 필터", before, len(filtered))

    if filters.get("release_date_from"):
        before = len(filtered)
        start = pd.to_datetime(filters["release_date_from"])
        filtered = filtered[filtered["release_date"] >= start]
        print_filter_step(log_rows, "출시일 시작 필터", before, len(filtered))

    if filters.get("release_date_to"):
        before = len(filtered)
        end = pd.to_datetime(filters["release_date_to"])
        filtered = filtered[filtered["release_date"] <= end]
        print_filter_step(log_rows, "출시일 종료 필터", before, len(filtered))

    # 3. 리뷰 조건 필터
    # 언어, Steam 라벨, 리뷰 작성일, 출시 기준 구간을 적용한다.
    if filters.get("languages") and "language" in filtered.columns:
        before = len(filtered)
        allowed = [x.lower() for x in filters["languages"]]
        filtered = filtered[filtered["language"].fillna("").str.lower().isin(allowed)]
        print_filter_step(log_rows, "언어 필터", before, len(filtered))

    if filters.get("steam_labels"):
        before = len(filtered)
        filtered = filtered[filtered["steam_label_text"].isin(filters["steam_labels"])]
        print_filter_step(log_rows, "Steam 라벨 필터", before, len(filtered))

    if filters.get("review_date_from"):
        before = len(filtered)
        start = pd.to_datetime(filters["review_date_from"])
        filtered = filtered[filtered["review_datetime"] >= start]
        print_filter_step(log_rows, "리뷰 작성일 시작 필터", before, len(filtered))

    if filters.get("review_date_to"):
        before = len(filtered)
        end = pd.to_datetime(filters["review_date_to"])
        filtered = filtered[filtered["review_datetime"] <= end]
        print_filter_step(log_rows, "리뷰 작성일 종료 필터", before, len(filtered))

    if filters.get("release_periods"):
        before = len(filtered)
        filtered = filtered[filtered["release_period"].isin(filters["release_periods"])]
        print_filter_step(log_rows, "출시 기준 리뷰 구간 필터", before, len(filtered))

    # 4. 리뷰 신뢰도 필터
    # 실제 구매 리뷰 위주로 보고, 무료 수령/얼리액세스 리뷰를 제외할 수 있다.
    if filters.get("steam_purchase_only") is True and "steam_purchase" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["steam_purchase"] == True]
        print_filter_step(log_rows, "Steam 구매 리뷰 필터", before, len(filtered))

    if filters.get("exclude_received_for_free") is True and "received_for_free" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["received_for_free"] != True]
        print_filter_step(log_rows, "무료 수령 리뷰 제외", before, len(filtered))

    if filters.get("exclude_early_access_reviews") is True and "written_during_early_access" in filtered.columns:
        before = len(filtered)
        filtered = filtered[filtered["written_during_early_access"] != True]
        print_filter_step(log_rows, "얼리액세스 리뷰 제외", before, len(filtered))

    return filtered.reset_index(drop=True), pd.DataFrame(log_rows)

# 분석 조건 적용
df_filtered, filter_log = apply_analysis_filters(df_candidates, ANALYSIS_FILTERS)

print("최종 필터링 리뷰 수:", len(df_filtered))
print("최종 필터링 게임 수:", df_filtered["appid"].nunique())

if RUN_CHECK_CELLS:
    display(filter_log)
    print("필터링 후 release_period 분포")
    display(df_filtered["release_period"].value_counts(dropna=False))
    print("필터링 후 Steam 라벨 분포")
    display(df_filtered["steam_label_text"].value_counts(dropna=False))


최소 리뷰 길이 필터: 166,875 -> 106,248 / 제거 60,627 (36.33%)
의미 있는 리뷰 필터: 106,248 -> 68,747 / 제거 37,501 (35.30%)
appid 필터: 68,747 -> 300 / 제거 68,447 (99.56%)
출시일 시작 필터: 300 -> 300 / 제거 0 (0.00%)
언어 필터: 300 -> 192 / 제거 108 (36.00%)
Steam 라벨 필터: 192 -> 192 / 제거 0 (0.00%)
Steam 구매 리뷰 필터: 192 -> 192 / 제거 0 (0.00%)
무료 수령 리뷰 제외: 192 -> 185 / 제거 7 (3.65%)
얼리액세스 리뷰 제외: 185 -> 185 / 제거 0 (0.00%)
최종 필터링 리뷰 수: 185
최종 필터링 게임 수: 1


,step,before_rows,after_rows,removed_rows,removed_rate
0,최소 리뷰 길이 필터,166875,106248,60627,0.363308
1,의미 있는 리뷰 필터,106248,68747,37501,0.352957
2,appid 필터,68747,300,68447,0.995636
3,출시일 시작 필터,300,300,0,0.000000
4,언어 필터,300,192,108,0.360000
5,Steam 라벨 필터,192,192,0,0.000000
6,Steam 구매 리뷰 필터,192,192,0,0.000000
7,무료 수령 리뷰 제외,192,185,7,0.036458
8,얼리액세스 리뷰 제외,185,185,0,0.000000


필터링 후 release_period 분포


release_period
D0-D30      91
D181+       46
D31-D90     30
D91-D180    18
Name: count, dtype: int64

필터링 후 Steam 라벨 분포


steam_label_text
positive    121
negative     64
Name: count, dtype: int64

# 5. 게임별 샘플링 및 LLM 입력 파일 저장

In [8]:
# 게임 1개에 대해 리뷰를 샘플링한다.
# - recent: 최신 리뷰 우선
# - random: 무작위
# - balanced_by_steam_label: 긍정/부정 리뷰를 가능하면 균형 있게 섞음
# - negative_heavy_by_steam_label: 부정 리뷰를 더 많이 포함하도록 섞음
def sample_one_game(group, reviews_per_game, mode, random_state):
    """
    게임 1개에 대해 리뷰를 샘플링하는 함수.
    샘플링 방식은 분석 목적에 따라 LLM 실행 파일에서 선택한다.
    """
    if reviews_per_game is None or len(group) <= reviews_per_game:
        return group

    if mode == "recent":
        return group.sort_values("review_datetime", ascending=False).head(reviews_per_game)

    if mode == "random":
        return group.sample(n=reviews_per_game, random_state=random_state)

    if mode == "balanced_by_steam_label":
        half = reviews_per_game // 2

        positive = group[group["steam_label_text"] == "positive"]
        negative = group[group["steam_label_text"] == "negative"]

        pos_n = min(len(positive), half)
        neg_n = min(len(negative), reviews_per_game - pos_n)

        pos_sample = positive.sample(n=pos_n, random_state=random_state) if pos_n > 0 else positive.head(0)
        neg_sample = negative.sample(n=neg_n, random_state=random_state) if neg_n > 0 else negative.head(0)

        sampled = pd.concat([pos_sample, neg_sample], axis=0)

        # 한쪽 라벨이 부족해서 목표 개수보다 적게 뽑힌 경우,
        # 부족한 수는 남은 리뷰에서 채운다.
        remain_n = reviews_per_game - len(sampled)
        if remain_n > 0:
            remain_pool = group.drop(index=sampled.index, errors="ignore")
            if len(remain_pool) > 0:
                add_n = min(remain_n, len(remain_pool))
                sampled = pd.concat([
                    sampled,
                    remain_pool.sample(n=add_n, random_state=random_state)
                ], axis=0)

        return sampled.sample(frac=1, random_state=random_state)

    if mode == "negative_heavy_by_steam_label":
        target_neg_n = int(np.ceil(reviews_per_game * NEGATIVE_SAMPLE_RATIO))

        positive = group[group["steam_label_text"] == "positive"]
        negative = group[group["steam_label_text"] == "negative"]

        neg_n = min(len(negative), target_neg_n)
        pos_n = min(len(positive), reviews_per_game - neg_n)

        neg_sample = negative.sample(n=neg_n, random_state=random_state) if neg_n > 0 else negative.head(0)
        pos_sample = positive.sample(n=pos_n, random_state=random_state) if pos_n > 0 else positive.head(0)

        sampled = pd.concat([neg_sample, pos_sample], axis=0)

        # 부정/긍정 리뷰가 부족해서 목표 개수보다 적게 뽑힌 경우,
        # 부족한 수는 남은 리뷰에서 채운다.
        remain_n = reviews_per_game - len(sampled)
        if remain_n > 0:
            remain_pool = group.drop(index=sampled.index, errors="ignore")
            if len(remain_pool) > 0:
                add_n = min(remain_n, len(remain_pool))
                sampled = pd.concat([
                    sampled,
                    remain_pool.sample(n=add_n, random_state=random_state)
                ], axis=0)

        return sampled.sample(frac=1, random_state=random_state)

    raise ValueError(f"지원하지 않는 SAMPLE_MODE입니다: {mode}")


# 게임별 샘플링 실행
def sample_reviews_per_game(df, reviews_per_game, mode, random_state):
    sampled_groups = []

    for _, group in df.groupby("appid", group_keys=False):
        sampled_groups.append(sample_one_game(group, reviews_per_game, mode, random_state))

    if not sampled_groups:
        return df.head(0)

    sampled = pd.concat(sampled_groups, axis=0).reset_index(drop=True)

    if MAX_TOTAL_REVIEWS is not None and len(sampled) > MAX_TOTAL_REVIEWS:
        sampled = sampled.sample(n=MAX_TOTAL_REVIEWS, random_state=random_state).reset_index(drop=True)

    if TEST_N is not None:
        sampled = sampled.head(TEST_N).copy()

    return sampled


df_sampled = sample_reviews_per_game(
    df_filtered,
    reviews_per_game=REVIEWS_PER_GAME,
    mode=SAMPLE_MODE,
    random_state=RANDOM_STATE,
)

# LLM에 전달할 텍스트 길이 제한은 LLM 실행 파일에서 적용한다.
df_sampled["review_text_for_llm"] = df_sampled["review_text_clean"].fillna("").astype(str).str.slice(0, MAX_REVIEW_CHARS)

# LLM 입력 파일 컬럼 구성
# 프롬프트에 필요한 리뷰/게임 정보와 후속 결과 매칭에 필요한 키만 남긴다.
LLM_INPUT_COLUMNS = [
    # 원본 리뷰 식별 정보
    "recommendationid",              # 리뷰 고유 ID입니다. Steam 리뷰 1개를 구분하는 식별자입니다.
    "appid",                         # Steam 게임 고유 ID입니다. 어떤 게임의 리뷰인지 구분할 때 사용합니다.
    "game_name",                     # 게임 이름입니다. appid만 보면 알아보기 어려우므로 함께 전달합니다.
    "language",                      # 리뷰 작성 언어입니다. 현재 분석에서는 영어 리뷰 필터링 여부 확인에 사용합니다.

    # 리뷰 작성 시점 / 출시 후 구간 정보
    "review_datetime",               # 리뷰 작성 일시입니다. 출시 후 어느 시점의 반응인지 확인할 때 사용합니다.
    "release_date",                  # 게임 출시일입니다. 리뷰 작성일과 비교해 출시 후 경과일을 계산할 때 사용합니다.
    "days_from_release",             # 출시일 기준 리뷰 작성일까지 지난 일수입니다.
    "release_period",                # 출시 후 기간 구간입니다. 예: D0-D7, D8-D30 등입니다.
    "release_period_detail",         # 출시 후 구간을 더 세부적으로 나눈 값입니다. 초기 반응을 더 자세히 볼 때 사용합니다.

    # Steam 원본 라벨 / 리뷰 메타 정보
    "steam_label_text",              # Steam 추천 여부를 positive/negative 같은 문자열로 바꾼 값입니다.
    "voted_up",                      # Steam 원본 추천 여부입니다. True면 추천, False면 비추천 리뷰입니다.
    "playtime_at_review_hours",      # 리뷰 작성 시점의 플레이타임입니다. 짧은 플레이 후 부정 리뷰인지 확인할 수 있습니다.
    "playtime_stage",                # 플레이타임을 구간화한 값입니다. 초반/중반/장기 플레이 리뷰를 구분할 때 사용합니다.
    "votes_up",                      # 해당 리뷰가 받은 '유용함' 투표 수입니다. 리뷰 영향력이나 신뢰도 참고용입니다.
    "weighted_vote_score",           # Steam에서 제공하는 리뷰 가중 점수입니다. 리뷰 노출/신뢰도 참고용입니다.
    "received_for_free",             # 무료로 받은 게임인지 여부입니다. 일반 구매자 반응과 구분할 때 사용합니다.
    "written_during_early_access",   # 얼리액세스 기간에 작성된 리뷰인지 여부입니다.

    # 게임 메타 정보
    "genres_text",                   # 게임 장르 목록을 문자열로 정리한 값입니다. 장르별 반응 분석에 사용합니다.
    "categories_text",               # 게임 카테고리/플레이 방식 목록입니다. 싱글/멀티/협동 여부 분석에 사용합니다.
    "top_steam_tags_text",           # 주요 Steam 태그 목록입니다. 태그 기반 유사 게임 분석이나 반응 비교에 사용합니다.

    # LLM 입력 텍스트 / 유효성 판단 정보
    "review_text_for_llm",           # LLM에 실제로 전달할 리뷰 본문입니다. 전처리된 텍스트를 사용합니다.
    "is_meaningful_review",          # 분석에 의미 있는 리뷰인지 여부입니다. 무의미한 리뷰를 제외하거나 별도 확인할 때 사용합니다.
    "meaningless_reason",            # 무의미한 리뷰로 판단된 이유입니다. 예: 짧은 문장, 이모지만 존재, 내용 없음 등입니다.
]

input_cols = [c for c in LLM_INPUT_COLUMNS if c in df_sampled.columns]
df_for_llm = df_sampled[input_cols].copy()

# 최종 LLM 입력 파일과 필터 로그를 저장한다.
# 이 파일을 보면 어떤 리뷰가 실제 분석 대상이 되었는지 재현할 수 있다.
df_for_llm.to_csv(LLM_INPUT_PATH, index=False, encoding="utf-8-sig")
filter_log.to_csv(FILTER_LOG_PATH, index=False, encoding="utf-8-sig")

# 게임별로 최종 샘플 리뷰 수와 라벨 분포를 요약한다.
sample_summary = (
    df_for_llm
    .groupby(["appid", "game_name"], as_index=False)
    .agg(
        sampled_review_count=("recommendationid", "count"),
        positive_count=("steam_label_text", lambda x: (x == "positive").sum()),
        negative_count=("steam_label_text", lambda x: (x == "negative").sum()),
        first_review_datetime=("review_datetime", "min"),
        last_review_datetime=("review_datetime", "max"),
    )
    .sort_values("sampled_review_count", ascending=False)
)
sample_summary.to_csv(SAMPLE_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("LLM 입력 리뷰 수:", len(df_for_llm))
print("LLM 입력 게임 수:", df_for_llm["appid"].nunique())
print("LLM 입력 저장:", LLM_INPUT_PATH)
print("필터 로그 저장:", FILTER_LOG_PATH)
print("샘플 요약 저장:", SAMPLE_SUMMARY_PATH)

display(df_for_llm.head())
display(sample_summary.head())


LLM 입력 리뷰 수: 185
LLM 입력 게임 수: 1
LLM 입력 저장: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_zoonomaly_v1\llm_input_reviews.csv
필터 로그 저장: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_zoonomaly_v1\llm_input_filter_log.csv
샘플 요약 저장: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_zoonomaly_v1\llm_input_sample_summary.csv


,recommendationid,appid,game_name,language,review_datetime,release_date,days_from_release,release_period,release_period_detail,steam_label_text,voted_up,playtime_at_review_hours,playtime_stage,votes_up,weighted_vote_score,received_for_free,written_during_early_access,genres_text,categories_text,top_steam_tags_text,review_text_for_llm,is_meaningful_review,meaningless_reason
0,159990344,2101890,Zoonomaly,english,2024-03-06 19:49:16,2024-03-06,0.0,D0-D30,D0-D7,positive,True,0.416667,very_early,2,0.447926,False,False,"Action, Adventure, Indie","Single-player, Family Sharing","Horror, Exploration, Puzzle, Action-Adventure, Open World, First-Person, Realistic, 3D, Action, Controller","The game is quite interesting, but there are nuances: optimization is not very good; graphics ; Please add Ukrainian...",True,meaningful
1,159991191,2101890,Zoonomaly,english,2024-03-06 20:04:52,2024-03-06,0.0,D0-D30,D0-D7,positive,True,0.233333,very_early,4,0.506971,False,False,"Action, Adventure, Indie","Single-player, Family Sharing","Horror, Exploration, Puzzle, Action-Adventure, Open World, First-Person, Realistic, 3D, Action, Controller","Just purchased and play this game. i love it no much. the creatures are so creepy and weird, just makes the scary ex...",True,meaningful
2,159992435,2101890,Zoonomaly,english,2024-03-06 20:27:11,2024-03-06,0.0,D0-D30,D0-D7,positive,True,0.283333,very_early,1,0.497487,False,False,"Action, Adventure, Indie","Single-player, Family Sharing","Horror, Exploration, Puzzle, Action-Adventure, Open World, First-Person, Realistic, 3D, Action, Controller","Good game, I had no trouble running the game at maximum settings. Although I do believe optimizations could be a bit...",True,meaningful
3,159992536,2101890,Zoonomaly,english,2024-03-06 20:28:58,2024-03-06,0.0,D0-D30,D0-D7,positive,True,0.500000,early,37,0.691098,False,False,"Action, Adventure, Indie","Single-player, Family Sharing","Horror, Exploration, Puzzle, Action-Adventure, Open World, First-Person, Realistic, 3D, Action, Controller","This is both terrifying and fun! I've been following lights are off on social media for a while, and I have to say t...",True,meaningful
4,159994367,2101890,Zoonomaly,english,2024-03-06 21:01:40,2024-03-06,0.0,D0-D30,D0-D7,positive,True,1.166667,early,0,0.500000,False,False,"Action, Adventure, Indie","Single-player, Family Sharing","Horror, Exploration, Puzzle, Action-Adventure, Open World, First-Person, Realistic, 3D, Action, Controller","The game does good at scaring you in some scenarios, there might be the need to adjust some monster agro and monster...",True,meaningful


,appid,game_name,sampled_review_count,positive_count,negative_count,first_review_datetime,last_review_datetime
0,2101890,Zoonomaly,185,121,64,2024-03-06 19:49:16,2026-04-29 01:57:49


# 6. PydanticAI 출력 스키마 정의

- 결과 컬럼이 매번 달라지는 것을 방지한다.
- JSON 파싱 오류를 줄인다.
- 후속 집계표를 안정적으로 만들 수 있다.
- `category` 목록은 이후 이슈 집계와 시각화의 기준이 된다.

| 설정값 | 의미 |
|---|---|
| `bug` | 버그 |
| `optimization` | 최적화 |
| `performance` | 성능/프레임 |
| `crash` | 튕김/실행 불가 |
| `control` | 조작감 |
| `balance` | 밸런스 |
| `difficulty` | 난이도 |
| `content_volume` | 콘텐츠 양 |
| `story` | 스토리 |
| `translation_localization` | 번역/현지화 |
| `ui_ux` | UI/UX |
| `price_value` | 가격 대비 가치 |
| `multiplayer_network` | 멀티/서버 |
| `save_progression` | 저장/진행도 |
| `graphics_audio` | 그래픽/사운드 |
| `gameplay_loop` | 핵심 재미/반복 구조 |
| `monetization` | 과금/DLC |
| `developer_communication` | 개발자 소통 |
| `positive_praise` | 전반적 칭찬 |
| `progression_grind` | 성장/노가다 |
| `other` | 기타 |

In [9]:
# ============================================================
# LLM 출력 스키마
# ============================================================
# PydanticAI의 output_type으로 사용할 Pydantic 모델이다.
# 이 스키마를 기준으로 LLM 출력이 구조화되어 들어온다.
# ============================================================

class IssueTag(BaseModel):
    category: Literal[
        "bug",                          # 버그, 오류, 비정상 동작
        "optimization",                 # 최적화 전반, 렉, 로딩, 프레임 저하
        "performance",                  # 성능, 사양, 프레임 관련 문제
        "crash",                        # 튕김, 실행 불가, 강제 종료
        "control",                      # 조작감, 키 설정, 컨트롤러 문제
        "balance",                      # 밸런스, 캐릭터/무기/시스템 불균형
        "difficulty",                   # 난이도 관련 불만/칭찬
        "content_volume",               # 콘텐츠 양 부족/풍부함
        "story",                        # 스토리, 서사, 캐릭터, 세계관
        "translation_localization",     # 번역, 현지화, 언어 지원 문제
        "ui_ux",                        # UI, UX, 메뉴, 정보 전달 문제
        "price_value",                  # 가격 대비 가치, 할인, 볼륨 대비 가격
        "multiplayer_network",          # 멀티플레이, 서버, 매칭, 네트워크
        "save_progression",             # 저장, 진행도, 체크포인트, 세이브 손실
        "graphics_audio",               # 그래픽, 사운드, 연출, 아트 스타일
        "gameplay_loop",                # 핵심 재미, 반복 구조, 전투/플레이 흐름
        "monetization",                 # 과금, DLC, BM, 유료 요소
        "developer_communication",      # 개발자 소통, 패치 대응, 공지
        "positive_praise",              # 구체 이슈라기보다 전반적 칭찬
        "progression_grind",            # 성장, 반복 플레이, 노가다 구조
        "other",                        # 위 범주로 분류하기 어려운 기타 이슈
    ] = Field(description="리뷰에서 언급된 세부 이슈 카테고리")

    # sentiment는 해당 이슈에 대한 감정 방향이다.
    # 리뷰 전체 감정이 아니라, 이 세부 이슈 하나에 대한 감정이다.
    sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="이 이슈에 대한 감정"
    )

    # evidence는 왜 이 카테고리/감정으로 판단했는지에 대한 짧은 근거다.
    # 원문 리뷰를 바탕으로 작성된다.
    evidence: str = Field(
        description="원문 리뷰를 바탕으로 한 짧은 판단 근거",
        min_length=1,
        max_length=160,
    )


class SteamReviewAnalysis(BaseModel):
    # 입력 리뷰 ID를 그대로 반환한다.
    recommendationid: str = Field(description="입력 리뷰 ID 그대로 반환")

    # LLM이 리뷰 본문만 보고 판단한 감정이다.
    # Steam의 voted_up과 다를 수 있다.
    llm_sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="리뷰 본문 기준 감정"
    )

    # 감정을 1~5점으로 수치화한 값이다.
    # 1은 매우 부정, 3은 중립, 5는 매우 긍정으로 해석한다.
    sentiment_score: int = Field(
        ge=1,
        le=5,
        description="1=매우 부정, 3=중립, 5=매우 긍정",
    )

    primary_issue: Literal[
        "bug",                          # 버그, 오류, 비정상 동작
        "optimization",                 # 최적화 전반, 렉, 로딩, 프레임 저하
        "performance",                  # 성능, 사양, 프레임 관련 문제
        "crash",                        # 튕김, 실행 불가, 강제 종료
        "control",                      # 조작감, 키 설정, 컨트롤러 문제
        "balance",                      # 밸런스, 캐릭터/무기/시스템 불균형
        "difficulty",                   # 난이도 관련 불만/칭찬
        "content_volume",               # 콘텐츠 양 부족/풍부함
        "story",                        # 스토리, 서사, 캐릭터, 세계관
        "translation_localization",     # 번역, 현지화, 언어 지원 문제
        "ui_ux",                        # UI, UX, 메뉴, 정보 전달 문제
        "price_value",                  # 가격 대비 가치, 할인, 볼륨 대비 가격
        "multiplayer_network",          # 멀티플레이, 서버, 매칭, 네트워크
        "save_progression",             # 저장, 진행도, 체크포인트, 세이브 손실
        "graphics_audio",               # 그래픽, 사운드, 연출, 아트 스타일
        "gameplay_loop",                # 핵심 재미, 반복 구조, 전투/플레이 흐름
        "monetization",                 # 과금, DLC, BM, 유료 요소
        "developer_communication",      # 개발자 소통, 패치 대응, 공지
        "positive_praise",              # 구체 이슈라기보다 전반적 칭찬
        "progression_grind",            # 성장, 반복 플레이, 노가다 구조
        "other",                        # 위 범주로 분류하기 어려운 기타 이슈
    ] = Field(description="리뷰의 대표 이슈")

    # 리뷰 안에서 발견된 여러 세부 이슈 목록이다.
    issue_tags: List[IssueTag] = Field(
        default_factory=list,
        description="리뷰에서 발견된 세부 이슈 목록",
    )

    # 개발사 입장에서 대응 긴급도다.
    urgency: Literal["low", "medium", "high"] = Field(
        description="개선 필요도 또는 대응 긴급도"
    )

    # 리뷰 핵심 내용 요약이다.
    summary: str = Field(
        description="리뷰 핵심 내용 요약",
        min_length=5,
        max_length=220,
    )

    # 개발사 관점의 후속 액션이다.
    suggested_action: str = Field(
        description="개발사 또는 운영자가 참고할 수 있는 개선 방향",
        min_length=5,
        max_length=260,
    )


class BatchSteamReviewAnalysis(BaseModel):
    """한 번의 배치 요청에서 여러 리뷰 결과를 받을 수 있도록 감싸는 모델."""
    results: List[SteamReviewAnalysis] = Field(
        description="리뷰별 분석 결과 목록"
    )


ISSUE_KR_MAP = {
    "bug": "버그",
    "optimization": "최적화",
    "performance": "성능",
    "crash": "크래시",
    "control": "조작감",
    "balance": "밸런스",
    "difficulty": "난이도",
    "content_volume": "콘텐츠 분량",
    "story": "스토리",
    "translation_localization": "번역/현지화",
    "ui_ux": "UI/UX",
    "price_value": "가격/가치",
    "multiplayer_network": "멀티/네트워크",
    "save_progression": "저장/진행",
    "graphics_audio": "그래픽/사운드",
    "gameplay_loop": "게임플레이 루프",
    "monetization": "과금",
    "developer_communication": "개발사 소통",
    "positive_praise": "긍정 칭찬",
    "progression_grind": "성장/반복 노가다",
    "other": "기타",
}


# 7. 프롬프트 및 PydanticAI Agent 설정
LLM에게 어떤 역할을 부여할지, 어떤 모델 설정으로 호출할지 정한다.

| 설정값 | 의미 |
|---|---|
| `system_prompt` | LLM에게 부여하는 분석 기준과 제한 사항 |
| `temperature=0.0` | 같은 리뷰에 대해 가능한 한 일관적인 분류가 나오도록 설정 |
| `review_agent` | PydanticAI가 Vertex AI Gemini를 호출할 때 사용하는 Agent |



In [10]:
system_prompt = """
당신은 Steam 인디게임 리뷰 분석 전문가입니다.

각 리뷰에 대해 다음을 판단하세요.
1. 리뷰 본문 기준 감정(llm_sentiment)
2. 가장 핵심적인 이슈(primary_issue)
3. 세부 이슈(issue_tags)
4. 개발사 관점 suggested_action
5. 개선 필요도 urgency

중요 규칙:
- recommendationid는 반드시 입력값 그대로 반환하세요.
- voted_up은 참고 정보일 뿐, 감정은 review 텍스트 기준으로 판단하세요.
- 추천 리뷰라도 불만이 많으면 mixed 또는 negative로 판단할 수 있습니다.
- 비추천 리뷰라도 장단점이 섞여 있으면 mixed로 판단할 수 있습니다.
- issue_tags에는 실제로 언급된 것만 넣으세요.
- evidence는 리뷰 본문에 근거가 있는 짧은 표현 또는 요약으로 작성하세요.
- review가 매우 짧거나 밈/농담 위주면 과잉 해석하지 마세요.
- 분석할 정보가 부족한 리뷰는 primary_issue를 other로 두고, urgency는 low로 판단하세요.
- 리뷰 본문에 근거가 없는 개선 제안은 작성하지 말고 보수적으로 작성하세요.
- 게임 메타데이터와 태그는 맥락 참고용이며, 리뷰 본문에 없는 내용을 억지로 추론하지 마세요.
- 개발사 개선 제안은 인디게임 개발사가 실제로 참고할 수 있게 구체적으로 작성하세요.
"""

# Gemini 모델 세부 설정
# temperature=0.0으로 두어 같은 리뷰에 대해 가능한 한 일관적인 분류가 나오도록 한다.
review_settings = GoogleModelSettings(
    temperature=0.0,
)

# Vertex 모델이 정상 생성된 경우에만 PydanticAI Agent를 만든다.
# RUN_LLM=False로 기존 결과만 읽을 때는 Agent가 없어도 후처리 셀을 볼 수 있다.
if vertex_model is not None:
    review_agent = Agent(
        vertex_model,
        output_type=BatchSteamReviewAnalysis,
        system_prompt=system_prompt,
        retries=MAX_RETRIES,
        output_retries=3,
    )
else:
    review_agent = None

print("PydanticAI Agent 생성 여부:", "O" if review_agent is not None else "X")


PydanticAI Agent 생성 여부: O


# 8. 프롬프트 생성 함수


In [11]:
def build_batch_prompt(batch_df):
    """
    여러 개의 리뷰를 한 번에 LLM에게 보내기 위한 프롬프트 생성 함수.
    """
    blocks = []

    for _, row in batch_df.iterrows():
        block = f"""
[REVIEW]
recommendationid: {row["recommendationid"]}
appid: {row["appid"]}
game_name: {row.get("game_name", "")}
genres: {row.get("genres_text", "")}
categories: {row.get("categories_text", "")}
steam_top_tags: {row.get("top_steam_tags_text", "")}
release_date: {row.get("release_date", "")}
review_datetime: {row.get("review_datetime", "")}
days_from_release: {row.get("days_from_release", "")}
release_period: {row.get("release_period", "")}
language: {row.get("language", "")}
voted_up: {row.get("voted_up", "")}
steam_label_text: {row.get("steam_label_text", "")}
playtime_at_review_hours: {row.get("playtime_at_review_hours", "")}
playtime_stage: {row.get("playtime_stage", "")}
votes_up: {row.get("votes_up", "")}
weighted_vote_score: {row.get("weighted_vote_score", "")}
received_for_free: {row.get("received_for_free", "")}
written_during_early_access: {row.get("written_during_early_access", "")}

review:
{row.get("review_text_for_llm", "")}
[/REVIEW]
"""
        blocks.append(block)

    prompt = (
        f"다음 {len(batch_df)}개의 Steam 리뷰를 각각 분석해주세요.\\n"
        "반드시 입력된 recommendationid를 그대로 유지해서 반환하세요.\\n"
        "결과는 지정된 Pydantic 스키마에 맞게 반환하세요.\\n\\n"
        + "\\n".join(blocks)
    )

    return prompt


# 9. PydanticAI Vertex LLM 호출 함수

실제 Vertex AI Gemini 호출을 수행하는 함수

핵심 흐름은 다음과 같다.
1. 리뷰 배치를 프롬프트로 변환한다.
2. PydanticAI Agent로 Vertex AI Gemini를 호출한다.
3. LLM이 반환한 결과를 `recommendationid` 기준으로 원본 리뷰와 매칭한다.
4. 성공/누락/실패 결과를 모두 기록한다.
5. 중간 결과를 checkpoint에 저장해 실행 중단 시에도 복구할 수 있게 한다.

AI와 씨름한 결과물이라 이게 맞는건지는....

In [12]:
# 동시에 실행될 LLM 요청 수를 제한하기 위한 Semaphore
# MAX_CONCURRENT=1이면 한 번에 요청 1개만 실행
sem = asyncio.Semaphore(MAX_CONCURRENT)

# 리뷰 배치 1개를 PydanticAI + Vertex AI Gemini로 분석
# 1. batch_df를 LLM 프롬프트로 변환
# 2. Vertex AI Gemini 호출
# 3. Pydantic 구조로 받은 결과를 원본 리뷰와 매칭
# 4. 결과를 all_results에 추가
# 5. 실패하면 재시도하고, 최종 실패 시 실패 기록을 남김
async def analyze_batch(batch_df, all_results, stats, pbar):
    """
    리뷰 배치 1개를 PydanticAI + Vertex AI Gemini로 분석한다.
    """
    async with sem:
        prompt = build_batch_prompt(batch_df)

        for attempt in range(MAX_RETRIES):
            try:
                if review_agent is None:
                    raise RuntimeError(
                        "review_agent가 생성되지 않았습니다. "
                        ".env의 GOOGLE_CLOUD_PROJECT, gcloud ADC 인증, pydantic-ai 설치 여부를 확인하세요."
                    )

                result = await review_agent.run(
                    prompt,
                    model_settings=review_settings,
                )

                # PydanticAI가 스키마에 맞춰 구조화한 결과를 가져온다.
                output = result.output
                output_items = getattr(output, "results", [])

                # 토큰 사용량을 누적해 예상 비용을 확인
                input_tokens, output_tokens = extract_usage_tokens(result)
                stats["input_tokens"] += input_tokens
                stats["output_tokens"] += output_tokens
                stats["requests"] += 1

                # 입력 리뷰 ID와 LLM 반환 ID를 비교해 누락/오반환 여부를 확인
                input_ids = set(batch_df["recommendationid"].astype(str).tolist())
                matched_ids = set()

                for item in output_items:
                    rid = str(item.recommendationid)

                    # LLM이 입력에 없던 ID를 반환하면 무시한다.
                    if rid not in input_ids:
                        continue

                    row = batch_df[batch_df["recommendationid"].astype(str) == rid].iloc[0]
                    matched_ids.add(rid)

                    record = {
                        # 중요
                        # "primary_issue": 부정/긍정 반응의 핵심 원인 분류입니다.
                        # "urgency": 개선 우선순위 판단에 사용합니다.
                        # "suggested_action": 패치/운영 방향 제안에 활용합니다.
                        "analysis_status": "success",  # LLM 분석이 정상적으로 완료된 리뷰임을 표시합니다.

                        # 원본 리뷰 식별 정보
                        "recommendationid": rid,  # 리뷰 고유 ID입니다. Steam 리뷰 1개를 구분하는 식별자입니다.
                        "appid": row.get("appid"),  # Steam 게임 고유 ID입니다. 어떤 게임의 리뷰인지 구분할 때 사용합니다.
                        "game_name": row.get("game_name", ""),  # 게임 이름입니다. appid만 보면 알아보기 어려우므로 함께 저장합니다.
                        "review_datetime": row.get("review_datetime", None),  # 리뷰 작성 일시입니다. 출시 후 반응 구간을 확인할 때 사용합니다.
                        "release_date": row.get("release_date", None),  # 게임 출시일입니다. 리뷰 작성 시점과 비교해 초기/장기 반응을 나눌 때 사용합니다.
                        "days_from_release": row.get("days_from_release", None),  # 출시일 기준 리뷰 작성일까지 지난 일수입니다.
                        "release_period": row.get("release_period", None),  # 출시 후 기간 구간입니다. 예: D0-D7, D8-D30 등입니다.

                        # Steam 라벨/리뷰 메타
                        "steam_label_text": row.get("steam_label_text", ""),  # Steam 추천 여부를 positive/negative 같은 문자열로 바꾼 값입니다.
                        "playtime_at_review_hours": row.get("playtime_at_review_hours", None),  # 리뷰 작성 시점의 플레이타임입니다. 짧은 플레이 후 부정 리뷰인지 확인할 수 있습니다.
                        "votes_up": row.get("votes_up", None),  # 해당 리뷰가 받은 '유용함' 투표 수입니다. 리뷰 영향력이나 신뢰도 참고용입니다.
                        "weighted_vote_score": row.get("weighted_vote_score", None),  # Steam에서 제공하는 리뷰 가중 점수입니다. 리뷰 노출/신뢰도 참고용입니다.

                        # LLM 분석 결과
                        "llm_sentiment": item.llm_sentiment,  # LLM이 판단한 리뷰 감정입니다. positive/negative/mixed/neutral 등으로 저장됩니다.
                        "sentiment_score": item.sentiment_score,  # LLM이 판단한 감정 점수입니다. 감정 강도를 수치로 비교할 때 사용합니다.
                        "primary_issue": item.primary_issue,  # LLM이 판단한 리뷰의 대표 이슈입니다. 버그/밸런스/콘텐츠/가격 등 주요 원인을 나타냅니다.
                        "issue_tags": pydantic_list_to_dicts(item.issue_tags),  # 리뷰 안에서 발견된 세부 이슈 태그 목록입니다. 한 리뷰에 여러 문제가 있을 수 있습니다.
                        "urgency": item.urgency,  # 개선 시급도입니다. 어떤 문제를 먼저 고쳐야 할지 우선순위 판단에 사용합니다.
                        "summary": item.summary,  # LLM이 요약한 리뷰 핵심 내용입니다. 원문을 빠르게 파악하기 위한 요약입니다.
                        "suggested_action": item.suggested_action,  # LLM이 제안한 개선 방향입니다. 패치/운영 방향 제안에 활용합니다.
                    }

                    all_results.append(record)

                # LLM이 누락한 리뷰가 있으면 누락 기록을 남긴다.
                missing_ids = input_ids - matched_ids
                for rid in missing_ids:
                    row = batch_df[batch_df["recommendationid"].astype(str) == rid].iloc[0]
                    all_results.append({
                        "analysis_status": "missing_in_llm_output",
                        "recommendationid": rid,
                        "appid": row.get("appid"),
                        "game_name": row.get("game_name", ""),
                        "steam_label_text": row.get("steam_label_text", ""),
                        "llm_sentiment": None,
                        "sentiment_score": None,
                        "primary_issue": None,
                        "issue_tags": [],
                        "urgency": None,
                        "summary": None,
                        "suggested_action": None,
                    })

                save_checkpoint(all_results)
                pbar.update(len(batch_df))
                return

            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    wait_sec = 2 ** attempt
                    print(f"배치 분석 실패, 재시도 {attempt + 1}/{MAX_RETRIES}: {e}")
                    await asyncio.sleep(wait_sec)
                else:
                    print(f"배치 최종 실패: {e}")

                    for _, row in batch_df.iterrows():
                        all_results.append({
                            "analysis_status": "failed",
                            "recommendationid": str(row.get("recommendationid")),
                            "appid": row.get("appid"),
                            "game_name": row.get("game_name", ""),
                            "steam_label_text": row.get("steam_label_text", ""),
                            "llm_sentiment": None,
                            "sentiment_score": None,
                            "primary_issue": None,
                            "issue_tags": [],
                            "urgency": None,
                            "summary": None,
                            "suggested_action": None,
                            "error_message": str(e),
                        })

                    save_checkpoint(all_results)
                    pbar.update(len(batch_df))
                    return

# 전체 LLM 분석을 실행한다.
# 처리 흐름:
# 1. 기존 checkpoint를 읽는다.
# 2. 현재 분석 대상 ID만 checkpoint에서 유지한다.
# 3. 이미 처리된 recommendationid는 제외한다.
# 4. 남은 리뷰를 BATCH_SIZE 단위로 나눈다.
# 5. CHUNK_SIZE 단위로 비동기 요청을 실행한다.
# 6. 중간 결과는 checkpoint에 계속 저장한다.
async def run_analysis(df):
    """
    전체 LLM 분석을 실행한다.
    """
    if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
        CHECKPOINT_PATH.unlink()
        print("기존 checkpoint 삭제:", CHECKPOINT_PATH)

    all_results = load_checkpoint()
    all_results = list(all_results)

    # 현재 분석 대상 ID만 checkpoint에서 유지한다.
    target_ids = set(df["recommendationid"].astype(str))
    all_results = [
        row for row in all_results
        if str(row.get("recommendationid")) in target_ids
    ]

    done_ids = {
        str(row.get("recommendationid"))
        for row in all_results
        if row.get("analysis_status") in ["success", "missing_in_llm_output", "failed"]
    }

    to_process = df[~df["recommendationid"].astype(str).isin(done_ids)].copy()

    stats = {
        "input_tokens": 0,
        "output_tokens": 0,
        "requests": 0,
        "checkpoint_count": len(done_ids),
        "to_process_count": len(to_process),
    }

    if len(to_process) == 0:
        print("새로 처리할 리뷰가 없습니다. checkpoint 또는 기존 결과를 사용합니다.")
        return all_results, stats

    batches = [
        to_process.iloc[i:i + BATCH_SIZE]
        for i in range(0, len(to_process), BATCH_SIZE)
    ]

    with tqdm(total=len(to_process), desc="PydanticAI Vertex 리뷰 분석 진행") as pbar:
        for start in range(0, len(batches), CHUNK_SIZE):
            chunk = batches[start:start + CHUNK_SIZE]

            tasks = [
                analyze_batch(batch_df, all_results, stats, pbar)
                for batch_df in chunk
            ]

            await asyncio.gather(*tasks)

            if REQUEST_SLEEP_SEC > 0:
                await asyncio.sleep(REQUEST_SLEEP_SEC)

    return all_results, stats


# 10. LLM 분석 실행

`RUN_LLM` 설정에 따라 실제 LLM 호출 여부가 달라진다.

- `RUN_LLM=True`: Vertex AI Gemini를 실제 호출한다.
- `RUN_LLM=False`: 기존 JSON/checkpoint 결과를 읽어 후처리만 수행한다.

실행 후에는 토큰 사용량과 예상 비용을 출력한다.

In [13]:
if RUN_LLM:
    results, stats = await run_analysis(df_for_llm)
else:
    results = load_existing_results()
    stats = {
        "input_tokens": 0,
        "output_tokens": 0,
        "requests": 0,
        "checkpoint_count": len(results),
        "to_process_count": 0,
    }

print_cost_report(
    input_tokens=stats["input_tokens"],
    output_tokens=stats["output_tokens"],
    requests=stats["requests"],
    checkpoint_count=stats["checkpoint_count"],
    to_process_count=stats["to_process_count"],
)

safe_results = to_serializable(results)
df_result_raw = pd.DataFrame(safe_results)

print("분석 결과 행 수:", len(df_result_raw))
display(df_result_raw.head())


PydanticAI Vertex 리뷰 분석 진행: 100%|██████████| 185/185 [03:22<00:00,  1.09s/it]

토큰 사용량 / 예상 비용
요청 수: 62
checkpoint에서 불러온 리뷰 수: 0
이번 실행에서 새로 처리할 리뷰 수: 185
입력 토큰: 112,049
출력 토큰: 36,902
예상 비용(USD): $0.125870
예상 비용(KRW): ₩189
분석 결과 행 수: 185


,analysis_status,recommendationid,appid,game_name,review_datetime,release_date,days_from_release,release_period,steam_label_text,playtime_at_review_hours,votes_up,weighted_vote_score,llm_sentiment,sentiment_score,primary_issue,issue_tags,urgency,summary,suggested_action
0,success,159990344,2101890,Zoonomaly,2024-03-06T19:49:16,2024-03-06T00:00:00,0.0,D0-D30,positive,0.416667,2.0,0.447926,mixed,3.0,optimization,"[{'category': 'optimization', 'sentiment': 'negative', 'evidence': 'optimization is not very good'}, {'category': 't...",medium,게임 자체는 흥미로우나 최적화가 아쉽고 우크라이나어 지원을 요청함.,"최적화 패치를 통해 프레임 드랍 및 성능 문제를 개선하고, 향후 업데이트 시 우크라이나어 현지화 추가를 고려하십시오."
1,success,159991191,2101890,Zoonomaly,2024-03-06T20:04:52,2024-03-06T00:00:00,0.0,D0-D30,positive,0.233333,4.0,0.506971,positive,5.0,positive_praise,"[{'category': 'positive_praise', 'sentiment': 'positive', 'evidence': 'creatures are so creepy and weird, best game ...",low,크리처 디자인과 공포 요소에 매우 만족하며 최고의 게임이라고 평가함.,"긍정적인 피드백에 감사하며, 커뮤니티 활성화를 위해 스팀 리뷰 작성 방법 등을 안내하는 공지를 게시하십시오."
2,success,159992435,2101890,Zoonomaly,2024-03-06T20:27:11,2024-03-06T00:00:00,0.0,D0-D30,positive,0.283333,1.0,0.497487,mixed,4.0,ui_ux,"[{'category': 'optimization', 'sentiment': 'negative', 'evidence': 'optimizations could be a bit better'}, {'categor...",medium,게임은 좋으나 인트로 텍스트 속도가 너무 빠르고 물리 버그가 존재함.,"인트로 텍스트 가독성을 위해 속도를 늦추거나 일시 정지 기능을 추가하고, 보고된 물리 엔진 버그를 수정하십시오."
3,success,159992536,2101890,Zoonomaly,2024-03-06T20:28:58,2024-03-06T00:00:00,0.0,D0-D30,positive,0.500000,37.0,0.691098,positive,5.0,positive_praise,"[{'category': 'positive_praise', 'sentiment': 'positive', 'evidence': 'terrifying and fun, amazing feeling to explor...",low,"게임의 분위기와 탐험 요소에 매우 만족하며, 개발사의 이전 활동을 지지하는 팬의 긍정적인 리뷰입니다.","현재의 긍정적인 분위기를 유지하며, 초기 플레이어들의 기대에 부응하는 콘텐츠 업데이트를 지속적으로 계획하세요."
4,success,159994367,2101890,Zoonomaly,2024-03-06T21:01:40,2024-03-06T00:00:00,0.0,D0-D30,positive,1.166667,0.0,0.500000,positive,4.0,balance,"[{'category': 'positive_praise', 'sentiment': 'positive', 'evidence': 'good at scaring, entertaining, simple but coo...",medium,"게임의 공포 요소와 메커니즘은 만족스러우나, 몬스터의 밸런스(속도 및 어그로) 조정이 필요하다는 의견입니다.",플레이어들이 언급한 몬스터의 어그로 범위와 이동 속도에 대한 밸런스 데이터를 수집하여 조정하는 패치를 고려하세요.


# 11. 리뷰 단위 결과 후처리 및 저장


In [14]:
def classify_sentiment_relation(row):
    """Steam 라벨과 LLM 감정의 관계 분류"""
    steam_label = row.get("steam_label_text")
    llm_sentiment = row.get("llm_sentiment")

    if steam_label == "positive":
        if llm_sentiment == "positive":
            return "exact_match"
        if llm_sentiment == "mixed":
            return "partial_match"
        if llm_sentiment == "negative":
            return "mismatch"
        return "unclear"

    if steam_label == "negative":
        if llm_sentiment == "negative":
            return "exact_match"
        if llm_sentiment == "mixed":
            return "partial_match"
        if llm_sentiment == "positive":
            return "mismatch"
        return "unclear"

    return "unknown"

# 리뷰 단위 결과 저장 컬럼
REVIEW_RESULT_COLUMNS = [
    # 분석 상태
    "analysis_status",               # LLM 분석 성공/실패 여부를 표시합니다.

    # 원본 리뷰 식별 정보
    "recommendationid",              # 리뷰 고유 ID입니다. Steam 리뷰 1개를 구분하는 식별자입니다.
    "appid",                         # Steam 게임 고유 ID입니다. 어떤 게임의 리뷰인지 구분할 때 사용합니다.
    "game_name",                     # 게임 이름입니다. appid만 보면 알아보기 어려우므로 함께 저장합니다.

    # 리뷰 작성 시점 / 출시 후 구간 정보
    "review_datetime",               # 리뷰 작성 일시입니다. 출시 후 반응 구간을 확인할 때 사용합니다.
    "release_date",                  # 게임 출시일입니다. 리뷰 작성 시점과 비교해 초기/장기 반응을 나눌 때 사용합니다.
    "days_from_release",             # 출시일 기준 리뷰 작성일까지 지난 일수입니다.
    "release_period",                # 출시 후 기간 구간입니다. 예: D0-D7, D8-D30 등입니다.

    # Steam 라벨 / 리뷰 메타 정보
    "steam_label_text",              # Steam 추천 여부를 positive/negative 같은 문자열로 바꾼 값입니다.
    "playtime_at_review_hours",      # 리뷰 작성 시점의 플레이타임입니다. 짧은 플레이 후 부정 리뷰인지 확인할 수 있습니다.
    "votes_up",                      # 해당 리뷰가 받은 '유용함' 투표 수입니다. 리뷰 영향력이나 신뢰도 참고용입니다.
    "weighted_vote_score",           # Steam에서 제공하는 리뷰 가중 점수입니다. 리뷰 노출/신뢰도 참고용입니다.

    # LLM 분석 결과
    "llm_sentiment",                 # LLM이 판단한 리뷰 감정입니다. positive/negative/mixed/neutral 등으로 저장됩니다.
    "sentiment_score",               # LLM이 판단한 감정 점수입니다. 감정 강도를 수치로 비교할 때 사용합니다.
    "primary_issue",                 # LLM이 판단한 리뷰의 대표 이슈입니다. 부정/긍정 반응의 핵심 원인 분류에 사용합니다.
    "issue_tags",                    # 리뷰 안에서 발견된 세부 이슈 태그 목록입니다. 한 리뷰에 여러 문제가 있을 수 있습니다.
    "urgency",                       # 개선 시급도입니다. 어떤 문제를 먼저 고쳐야 할지 우선순위 판단에 사용합니다.
    "summary",                       # LLM이 요약한 리뷰 핵심 내용입니다. 원문을 빠르게 파악하기 위한 요약입니다.
    "suggested_action",              # LLM이 제안한 개선 방향입니다. 패치/운영 방향 제안에 활용합니다.

    # Steam 라벨과 LLM 판단 비교 결과
    "sentiment_relation",            # Steam 추천/비추천 라벨과 LLM 감정 판단이 어느 정도 일치하는지 비교한 값입니다.
]

# 결과 후처리
# df_result_all: 분석 성공/실패 상태를 모두 포함한 메모리용 결과
# df_result: 보고서에서 사용할 성공 분석 결과만 포함한 메모리용 결과
# df_result_save: CSV 저장용 최소 컬럼 결과
def select_existing_columns(df, columns):
    """요청한 컬럼 중 실제 존재하는 컬럼만 선택한다."""
    return [col for col in columns if col in df.columns]


if len(df_result_raw) > 0:
    df_result_all = df_result_raw.copy()

    for col in ["review_datetime", "release_date"]:
        if col in df_result_all.columns:
            df_result_all[col] = pd.to_datetime(df_result_all[col], errors="coerce")

    if {"steam_label_text", "llm_sentiment"}.issubset(df_result_all.columns):
        df_result_all["sentiment_relation"] = df_result_all.apply(classify_sentiment_relation, axis=1)
    else:
        df_result_all["sentiment_relation"] = "unknown"

    if "analysis_status" in df_result_all.columns:
        df_result = df_result_all[df_result_all["analysis_status"] == "success"].copy()
        df_failed_log = df_result_all[df_result_all["analysis_status"] != "success"].copy()
    else:
        df_result = df_result_all.copy()
        df_failed_log = pd.DataFrame()

    if SAVE_RESULT_JSON:
        safe_success_results = to_serializable(df_result.to_dict(orient="records"))
        with open(RESULT_JSON_PATH, "w", encoding="utf-8") as f:
            json.dump(safe_success_results, f, ensure_ascii=False, indent=2)
        print("리뷰 단위 LLM 결과 JSON 저장:", RESULT_JSON_PATH)

    review_cols = select_existing_columns(df_result, REVIEW_RESULT_COLUMNS)
    df_result_save = df_result[review_cols].copy()
    df_result_save.to_csv(RESULT_CSV_PATH, index=False, encoding="utf-8-sig")

    print("리뷰 단위 LLM 결과 CSV 저장:", RESULT_CSV_PATH)
    print("CSV 저장 컬럼 수:", len(review_cols))
    print("성공 분석 리뷰 수:", len(df_result))
    print("실패/누락 리뷰 수:", len(df_failed_log))
else:
    df_result_all = pd.DataFrame()
    df_result = pd.DataFrame()
    df_result_save = pd.DataFrame(columns=REVIEW_RESULT_COLUMNS)
    df_failed_log = pd.DataFrame()

    print("분석 결과가 없습니다.")


리뷰 단위 LLM 결과 JSON 저장: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_zoonomaly_v1\llm_review_analysis_result.json
리뷰 단위 LLM 결과 CSV 저장: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_zoonomaly_v1\llm_review_analysis_result.csv
CSV 저장 컬럼 수: 20
성공 분석 리뷰 수: 181
실패/누락 리뷰 수: 4


# 12. 이슈 태그 펼치기


In [15]:
# 이슈 태그 펼친 결과 저장 컬럼
ISSUE_TAG_FLAT_COLUMNS = [
    "recommendationid",
    "appid",
    "game_name",
    "steam_label_text",
    "llm_sentiment",
    "primary_issue",
    "urgency",
    "release_period",
    "playtime_at_review_hours",
    "votes_up",
    "weighted_vote_score",
    "tag_category",
    "issue_name_kor",
    "tag_sentiment",
    "tag_evidence",
]


def flatten_issue_tags(df):
    """
    리뷰별 issue_tags 리스트를 이슈 단위 행으로 펼친다.
    """
    flat_rows = []

    if len(df) == 0 or "issue_tags" not in df.columns:
        return pd.DataFrame(columns=ISSUE_TAG_FLAT_COLUMNS)

    for _, row in df.iterrows():
        tags = row.get("issue_tags", [])

        if isinstance(tags, str):
            try:
                tags = json.loads(tags)
            except Exception:
                tags = []

        if not isinstance(tags, list):
            continue

        for tag in tags:
            if not isinstance(tag, dict):
                continue

            category = tag.get("category")
            issue_name_kor = ISSUE_KR_MAP.get(category, category)

            flat_rows.append({
                "recommendationid": row.get("recommendationid"),
                "appid": row.get("appid"),
                "game_name": row.get("game_name"),
                "steam_label_text": row.get("steam_label_text"),
                "llm_sentiment": row.get("llm_sentiment"),
                "primary_issue": row.get("primary_issue"),
                "urgency": row.get("urgency"),
                "release_period": row.get("release_period"),
                "playtime_at_review_hours": row.get("playtime_at_review_hours"),
                "votes_up": row.get("votes_up"),
                "weighted_vote_score": row.get("weighted_vote_score"),
                "tag_category": category,
                "issue_name_kor": issue_name_kor,
                "tag_sentiment": tag.get("sentiment"),
                "tag_evidence": tag.get("evidence"),
            })

    return pd.DataFrame(flat_rows, columns=ISSUE_TAG_FLAT_COLUMNS)


df_issue_tags_flat = flatten_issue_tags(df_result)
df_issue_tags_flat.to_csv(ISSUE_TAG_FLAT_PATH, index=False, encoding="utf-8-sig")

print("이슈 태그 펼친 결과 저장:", ISSUE_TAG_FLAT_PATH)
print("이슈 태그 행 수:", len(df_issue_tags_flat))
display(df_issue_tags_flat.head())


이슈 태그 펼친 결과 저장: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_zoonomaly_v1\llm_issue_tags_flat.csv
이슈 태그 행 수: 378


,recommendationid,appid,game_name,steam_label_text,llm_sentiment,primary_issue,urgency,release_period,playtime_at_review_hours,votes_up,weighted_vote_score,tag_category,issue_name_kor,tag_sentiment,tag_evidence
0,159990344,2101890,Zoonomaly,positive,mixed,optimization,medium,D0-D30,0.416667,2.0,0.447926,optimization,최적화,negative,optimization is not very good
1,159990344,2101890,Zoonomaly,positive,mixed,optimization,medium,D0-D30,0.416667,2.0,0.447926,translation_localization,번역/현지화,neutral,Please add Ukrainian localization
2,159990344,2101890,Zoonomaly,positive,mixed,optimization,medium,D0-D30,0.416667,2.0,0.447926,positive_praise,긍정 칭찬,positive,The game is quite interesting
3,159991191,2101890,Zoonomaly,positive,positive,positive_praise,low,D0-D30,0.233333,4.0,0.506971,positive_praise,긍정 칭찬,positive,"creatures are so creepy and weird, best game ever"
4,159992435,2101890,Zoonomaly,positive,mixed,ui_ux,medium,D0-D30,0.283333,1.0,0.497487,optimization,최적화,negative,optimizations could be a bit better


# 13. 선택 요약 파일 생성
기본 보고서에는 필수 파일이 아니므로 `SAVE_OPTIONAL_SUMMARY_FILES=True`일 때만 저장한

필요하면 보고서 노트북에서 `df_result`, `df_issue_tags_flat`을 이용해 다시 집계 가능

In [16]:
if SAVE_OPTIONAL_SUMMARY_FILES:
    game_summary = (
        df_result
        .groupby(["appid", "game_name"], as_index=False)
        .agg(
            analyzed_review_count=("recommendationid", "count"),
            steam_positive_count=("steam_label_text", lambda x: (x == "positive").sum()),
            steam_negative_count=("steam_label_text", lambda x: (x == "negative").sum()),
            llm_positive_count=("llm_sentiment", lambda x: (x == "positive").sum()),
            llm_negative_mixed_count=("llm_sentiment", lambda x: x.isin(["negative", "mixed"]).sum()),
            main_llm_sentiment=("llm_sentiment", lambda x: x.dropna().value_counts().idxmax() if len(x.dropna()) else None),
            main_issue=("primary_issue", lambda x: x.dropna().value_counts().idxmax() if len(x.dropna()) else None),
            high_urgency_count=("urgency", lambda x: (x == "high").sum()),
        )
    )

    game_summary["steam_positive_rate_sample"] = np.where(
        game_summary["analyzed_review_count"] > 0,
        game_summary["steam_positive_count"] / game_summary["analyzed_review_count"],
        np.nan,
    )
    game_summary["llm_positive_rate"] = np.where(
        game_summary["analyzed_review_count"] > 0,
        game_summary["llm_positive_count"] / game_summary["analyzed_review_count"],
        np.nan,
    )
    game_summary["high_urgency_rate"] = np.where(
        game_summary["analyzed_review_count"] > 0,
        game_summary["high_urgency_count"] / game_summary["analyzed_review_count"],
        np.nan,
    )

    game_summary.to_csv(GAME_SUMMARY_PATH, index=False, encoding="utf-8-sig")
    print("게임 단위 요약 저장:", GAME_SUMMARY_PATH)

    if len(df_issue_tags_flat) > 0:
        issue_sentiment_summary = (
            df_issue_tags_flat
            .groupby("tag_category", as_index=False)
            .agg(
                total_count=("tag_category", "count"),
                positive_count=("tag_sentiment", lambda x: (x == "positive").sum()),
                negative_count=("tag_sentiment", lambda x: (x == "negative").sum()),
                neutral_count=("tag_sentiment", lambda x: (x == "neutral").sum()),
                mixed_count=("tag_sentiment", lambda x: (x == "mixed").sum()),
                affected_review_count=("recommendationid", "nunique"),
                representative_evidence=("tag_evidence", lambda s: " / ".join(s.dropna().astype(str).head(3))),
            )
        )

        issue_sentiment_summary["negative_rate"] = np.where(
            issue_sentiment_summary["total_count"] > 0,
            issue_sentiment_summary["negative_count"] / issue_sentiment_summary["total_count"],
            np.nan,
        )
        issue_sentiment_summary["positive_rate"] = np.where(
            issue_sentiment_summary["total_count"] > 0,
            issue_sentiment_summary["positive_count"] / issue_sentiment_summary["total_count"],
            np.nan,
        )
        issue_sentiment_summary["issue_name_kor"] = (
            issue_sentiment_summary["tag_category"].map(ISSUE_KR_MAP).fillna(issue_sentiment_summary["tag_category"])
        )
        issue_sentiment_summary["issue_name"] = issue_sentiment_summary["issue_name_kor"]
    else:
        issue_sentiment_summary = pd.DataFrame()

    issue_sentiment_summary.to_csv(ISSUE_PRIORITY_PATH, index=False, encoding="utf-8-sig")
    print("이슈 우선순위 요약 저장:", ISSUE_PRIORITY_PATH)
else:
    print("선택 요약 파일 저장 생략: SAVE_OPTIONAL_SUMMARY_FILES=False")


선택 요약 파일 저장 생략: SAVE_OPTIONAL_SUMMARY_FILES=False


# 14. 산출물 점검


In [17]:
output_check_targets = {
    "LLM 입력 리뷰 CSV": LLM_INPUT_PATH,
    "리뷰별 LLM 분석 결과 CSV": RESULT_CSV_PATH,
    "이슈 태그 펼친 결과 CSV": ISSUE_TAG_FLAT_PATH,
    "LLM 분석 중간 저장 파일": CHECKPOINT_PATH,
}

if SAVE_RESULT_JSON:
    output_check_targets["리뷰별 LLM 분석 결과 JSON"] = RESULT_JSON_PATH

if SAVE_OPTIONAL_SUMMARY_FILES:
    output_check_targets["게임 단위 요약표 CSV"] = GAME_SUMMARY_PATH
    output_check_targets["이슈 우선순위 요약 CSV"] = ISSUE_PRIORITY_PATH

output_check_rows = []

for name, path in output_check_targets.items():
    exists = path.exists()
    rows = None
    columns = None

    if exists and path.suffix.lower() == ".csv":
        try:
            temp_df = pd.read_csv(path)
            rows = len(temp_df)
            columns = len(temp_df.columns)
        except Exception:
            rows = "읽기 실패"
            columns = "읽기 실패"

    output_check_rows.append({
        "산출물": name,
        "exists": exists,
        "rows": rows,
        "columns": columns,
        "path": str(path),
    })

output_check = pd.DataFrame(output_check_rows)
display(output_check)


,산출물,exists,rows,columns,path
0,LLM 입력 리뷰 CSV,True,185.0,23.0,C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_zoonomaly_v1\llm_input_reviews.csv
1,리뷰별 LLM 분석 결과 CSV,True,181.0,20.0,C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_zoonomaly_v1\llm_review_analysis_r...
2,이슈 태그 펼친 결과 CSV,True,378.0,15.0,C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_zoonomaly_v1\llm_issue_tags_flat.csv
3,LLM 분석 중간 저장 파일,True,NaN,NaN,C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_zoonomaly_v1\llm_review_analysis_c...
4,리뷰별 LLM 분석 결과 JSON,True,NaN,NaN,C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_zoonomaly_v1\llm_review_analysis_r...


# 15. 출력 테이블 설명

`llm_review_analysis_result.csv` 컬럼 명세
| 컬럼명                        | 타입        | 설명                      |
| -------------------------- | --------- | ----------------------- |
| `analysis_status`          | str       | LLM 분석 성공/실패 여부         |
| `recommendationid`         | int       | Steam 리뷰 고유 ID          |
| `appid`                    | int       | Steam 게임 고유 ID          |
| `game_name`                | str       | 게임 이름                   |
| `review_datetime`          | datetime  | 리뷰 작성 일시                |
| `release_date`             | datetime  | 게임 출시일                  |
| `days_from_release`        | float     | 출시일 기준 리뷰 작성일까지 지난 일수   |
| `release_period`           | str       | 출시 후 리뷰 작성 구간           |
| `steam_label_text`         | str       | Steam 추천/비추천 라벨         |
| `playtime_at_review_hours` | float     | 리뷰 작성 시점 플레이타임          |
| `votes_up`                 | int       | 리뷰가 받은 유용함 투표 수         |
| `weighted_vote_score`      | float     | Steam 리뷰 가중 점수          |
| `llm_sentiment`            | str       | LLM이 판단한 리뷰 전체 감정       |
| `sentiment_score`          | int       | LLM이 판단한 감정 점수          |
| `primary_issue`            | str       | 리뷰의 대표 이슈               |
| `issue_tags`               | list/dict | 리뷰 안에서 발견된 세부 이슈 태그 목록  |
| `urgency`                  | str       | 개선 시급도                  |
| `summary`                  | str       | LLM이 요약한 리뷰 핵심 내용       |
| `suggested_action`         | str       | LLM이 제안한 개선 방향          |
| `sentiment_relation`       | str       | Steam 라벨과 LLM 감정 판단의 관계 |

`llm_issue_tags_flat.csv` 컬럼 명세
| 컬럼명                        | 타입    | 설명                    |
| -------------------------- | ----- | --------------------- |
| `recommendationid`         | int   | Steam 리뷰 고유 ID        |
| `appid`                    | int   | Steam 게임 고유 ID        |
| `game_name`                | str   | 게임 이름                 |
| `steam_label_text`         | str   | Steam 추천/비추천 라벨       |
| `llm_sentiment`            | str   | LLM이 판단한 리뷰 전체 감정     |
| `primary_issue`            | str   | 리뷰의 대표 이슈             |
| `urgency`                  | str   | 개선 시급도                |
| `release_period`           | str   | 출시 후 리뷰 작성 구간         |
| `playtime_at_review_hours` | float | 리뷰 작성 시점 플레이타임        |
| `votes_up`                 | int   | 리뷰가 받은 유용함 투표 수       |
| `weighted_vote_score`      | float | Steam 리뷰 가중 점수        |
| `tag_category`             | str   | 세부 이슈 카테고리            |
| `issue_name_kor`           | str   | 세부 이슈의 한국어 이름         |
| `tag_sentiment`            | str   | 세부 이슈에 대한 감정          |
| `tag_evidence`             | str   | LLM이 해당 태그를 판단한 근거 문장 |


# 16. 초간단 결과 점검

In [18]:
df = df_result.copy()
tag = df_issue_tags_flat.copy()

print("리뷰 수:", len(df))
print("게임 수:", df["appid"].nunique())
print("이슈 태그 수:", len(tag))

print("\n[분석 성공 여부]")
print(df["analysis_status"].value_counts())

print("\n[출시 구간]")
print(df["release_period"].value_counts())

print("\n[Steam 라벨]")
print(df["steam_label_text"].value_counts(normalize=True).round(3) * 100)

print("\n[LLM 감정]")
print(df["llm_sentiment"].value_counts(normalize=True).round(3) * 100)

print("\n[긴급도]")
print(df["urgency"].value_counts(normalize=True).round(3) * 100)

print("\n[Steam 라벨 - LLM 감정 관계]")
print(df["sentiment_relation"].value_counts(normalize=True).round(3) * 100)

print("\n[대표 이슈 Top 10]")
print(df["primary_issue"].value_counts().head(10))

print("\n[세부 이슈 Top 10]")
print(tag["tag_category"].value_counts().head(10))

print("\n[필수 컬럼 결측치]")
check_cols = [
    "analysis_status", "recommendationid", "appid", "game_name",
    "steam_label_text", "llm_sentiment", "primary_issue",
    "urgency", "summary", "suggested_action", "sentiment_relation"
]
print(df[check_cols].isna().sum())

print("\n[태그 근거 빈 값 개수]")
print(tag["tag_evidence"].isna().sum() + (tag["tag_evidence"].astype(str).str.strip() == "").sum())

리뷰 수: 181
게임 수: 1
이슈 태그 수: 378

[분석 성공 여부]
analysis_status
success    181
Name: count, dtype: int64

[출시 구간]
release_period
D0-D30      91
D181+       45
D31-D90     27
D91-D180    18
Name: count, dtype: int64

[Steam 라벨]
steam_label_text
positive    64.6
negative    35.4
Name: proportion, dtype: float64

[LLM 감정]
llm_sentiment
positive    47.5
negative    36.5
mixed       14.9
neutral      1.1
Name: proportion, dtype: float64

[긴급도]
urgency
low       35.9
medium    32.0
high      32.0
Name: proportion, dtype: float64

[Steam 라벨 - LLM 감정 관계]
sentiment_relation
exact_match      81.2
partial_match    14.9
mismatch          2.8
unclear           1.1
Name: proportion, dtype: float64

[대표 이슈 Top 10]
primary_issue
positive_praise     48
gameplay_loop       47
difficulty          19
bug                 16
ui_ux                7
control              7
other                7
performance          7
graphics_audio       5
save_progression     5
Name: count, dtype: int64

[세부 이슈 Top 10]
tag_catego